# MoE-LoRA (OpenMOSE RWKV-LM-RLHF) sobre RWKV-7 1.5B en Kaggle 1xT4
- Modelo base: Kateto RWKV-7 1.5B World v2 (kateto-v7-sft/rwkv-0.pth, L=24, D=2048, FFN=8192).
- Dataset: Kateto ORPO v8 (550 pares convertidos al formato SFT h5 de OpenMOSE).
- Arquitectura MoE: 4 expertos FFN (rank 32), top-2 activos por token con router balanceado.
- Guardas: verificación de checkpoint .pth, claves expert/router y ausencia de NaN.
- Métrica clave: s_por_paso_real (throughput) y convergencia con loss finita.

In [ ]:
# Celda 0b - tee: todo stdout/err a /kaggle/working/run.log para que el output lo traiga
import sys, os
class _Tee:
    def __init__(self, *fs): self.fs = fs
    def write(self, s):
        for f in self.fs:
            try: f.write(s); f.flush()
            except Exception: pass
    def flush(self):
        for f in self.fs:
            try: f.flush()
            except Exception: pass
    def isatty(self): return False
_f = open('/kaggle/working/run.log', 'a', buffering=1)
sys.stdout = _Tee(sys.__stdout__, _f)
sys.stderr = _Tee(sys.__stderr__, _f)
print('[tee] log -> /kaggle/working/run.log')
import traceback
traceback.print_exc = traceback.print_exc


In [ ]:
print('[STEP] cell2 start', flush=True)
import os
import sys
import subprocess
import shutil
import glob
import time

print("=== Celda 1: Setup y Clonado de RWKV-LM-RLHF ===")
subprocess.run(["nvidia-smi"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pytorch-lightning==2.4.0", "bitsandbytes", "h5py", "einops", "prompt_toolkit", "wandb"
], check=True)

repo_dir = "/kaggle/working/RWKV-LM-RLHF"
if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/OpenMOSE/RWKV-LM-RLHF", repo_dir], check=True)
    fetch_proc = subprocess.run(
        ["git", "-C", repo_dir, "fetch", "--depth", "1", "origin", "f58064b059c0dd4d29e7723efccab1598d66a6e7"],
        capture_output=True, text=True
    )
    if fetch_proc.returncode == 0:
        subprocess.run(["git", "-C", repo_dir, "checkout", "FETCH_HEAD"], check=True)
        print("Checked out pinned commit f58064b059c0dd4d29e7723efccab1598d66a6e7")
    else:
        print(f"Warning: git fetch failed ({fetch_proc.stderr.strip()}), continuing with default HEAD")

import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    print(f"VRAM Total: {vram_bytes / (1024**3):.2f} GB ({vram_bytes} bytes)")
else:
    print("WARNING: CUDA not available!")

print('[STEP] cell2 end', flush=True)


In [ ]:
print('[STEP] cell3 start', flush=True)
# Celda 1b - shim lib2to3: Python 3.13 lo elimino y src/datasets/binidx.py lo importa
import os, sys
pkg = '/kaggle/working/shim/lib2to3/pgen2'
os.makedirs(pkg, exist_ok=True)
open('/kaggle/working/shim/lib2to3/__init__.py','w').write('')
open(os.path.join(pkg,'__init__.py'),'w').write('')
open(os.path.join(pkg,'token.py'),'w').write('from token import *\n')
sys.path.insert(0, '/kaggle/working/shim')
os.environ['PYTHONPATH'] = '/kaggle/working/shim' + (':' + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')
import lib2to3.pgen2.token as _shimtok
print('[shim] lib2to3.pgen2.token OK, NAME =', _shimtok.NAME)

print('[STEP] cell3 end', flush=True)


In [ ]:
print('[STEP] cell4 start', flush=True)
import os
import glob
import json
import math

print("=== Celda 2: Preparación de Datos ===")
candidates = glob.glob("/kaggle/input/**/*orpo*.jsonl", recursive=True)
v8_candidates = [f for f in candidates if "v8" in os.path.basename(f)]
if not v8_candidates:
    v8_candidates = [f for f in candidates if "v8" in f]

assert v8_candidates, f"No se encontró dataset ORPO v8 en /kaggle/input/. Encontrados: {candidates}"
src_dataset = v8_candidates[0]
print(f"Dataset seleccionado: {src_dataset}")

out_folder = "/kaggle/working/data"
os.makedirs(out_folder, exist_ok=True)
sft_file = os.path.join(out_folder, "sft_text.jsonl")

rows = []
with open(src_dataset, "r", encoding="utf-8") as f:
    for line in f:
        l = line.strip()
        if l:
            rows.append(json.loads(l))

assert len(rows) == 550, f"Expected 550 rows, got {len(rows)}"

n_chosen_sin_im_end = 0
lengths = []

with open(sft_file, "w", encoding="utf-8") as f_out:
    for idx, r in enumerate(rows):
        assert "prompt" in r and "chosen" in r and "rejected" in r, f"Fila {idx} missing prompt/chosen/rejected"
        prompt = r["prompt"]
        chosen = r["chosen"]
        if not chosen.endswith("<|im_end|>"):
            n_chosen_sin_im_end += 1
        text = prompt + chosen
        lengths.append(len(text))
        f_out.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")

sorted_lens = sorted(lengths)
k_p95 = math.ceil((len(sorted_lens) - 1) * 0.95)
chars_p95 = sorted_lens[k_p95]
chars_max = sorted_lens[-1]

print("Reporte de datos:")
print(f"  n: {len(rows)}")
print(f"  chosen sin <|im_end|>: {n_chosen_sin_im_end}")
print(f"  chars p95: {chars_p95}")
print(f"  chars max: {chars_max}")
print(f"Archivo SFT generado: {sft_file} ({os.path.getsize(sft_file)} bytes)")

print('[STEP] cell4 end', flush=True)


In [ ]:
print('[STEP] cell5 start', flush=True)
import os
import sys
import subprocess
import h5py
import numpy as np

print("=== Celda 3: Generación de sft.h5 ===")
cwd_main = "/kaggle/working/RWKV-LM-RLHF/main"
h5_target = "/kaggle/working/sft.h5"

cmd_h5 = [
    sys.executable, "sft_generate_h5.py",
    "--input_folder", "/kaggle/working/data",
    "--output_parquet", h5_target
]

print(f"Ejecutando en {cwd_main}: {' '.join(cmd_h5)}")
res_h5 = subprocess.run(cmd_h5, cwd=cwd_main, capture_output=True, text=True)
if res_h5.returncode != 0:
    print("STDOUT:", res_h5.stdout[-1500:] if len(res_h5.stdout) > 1500 else res_h5.stdout)
    print("STDERR:", res_h5.stderr)
    raise RuntimeError(f"sft_generate_h5.py falló con código {res_h5.returncode}")

assert os.path.exists(h5_target), f"No se encontró {h5_target}"

with h5py.File(h5_target, "r") as f:
    assert "tokens" in f, "Clave 'tokens' ausente en h5"
    n_tokens = len(f["tokens"])
    lens = [len(f["tokens"][i]) for i in range(n_tokens)]
    med_tokens = float(np.median(lens)) if lens else 0.0

print("sft.h5 verificado exitosamente:")
print(f"  len(f['tokens']): {n_tokens}")
print(f"  mediana longitud tokens: {med_tokens:.1f}")

assert n_tokens >= 500, f"Assert len(f['tokens']) >= 500 falló: {n_tokens}"

print('[STEP] cell5 end', flush=True)


In [ ]:
import glob
base_candidates = glob.glob("/kaggle/input/**/rwkv-0.pth", recursive=True)
if not base_candidates:
    base_candidates = glob.glob("/kaggle/input/**/*.pth", recursive=True)
assert base_candidates, "No se encontro modelo base rwkv-0.pth en /kaggle/input"
base_model_path = base_candidates[0]
print(f"Modelo base resuelto: {base_model_path} ({os.path.getsize(base_model_path)} bytes)")

# Fix medido: train.py:670 hace print(trainer.strategy.config) y con pytorch-lightning 2.4 +
# SingleDeviceStrategy eso revienta con AttributeError ANTES de trainer.fit(). Se reemplaza la
# linea por un print inofensivo (una sola linea, misma indentacion: un try/except multilinea ya
# rompio el archivo con IndentationError en la v5).
_tr = "/kaggle/working/RWKV-LM-RLHF/main/train.py"
_s = open(_tr, encoding="utf-8").read()
_s = _s.replace(
    "print(trainer.strategy.config)",
    "print('[patch] strategy.config omitido: Lightning 2.4 no lo expone')",
)
open(_tr, "w", encoding="utf-8").write(_s)
print("[patch] train.py:670 reemplazado por print seguro")
import py_compile, tempfile
py_compile.compile(_tr, cfile=tempfile.mktemp(), doraise=True)
print("[patch] train.py compila OK")

argv = [
    sys.executable, "train.py",
    "--load_model", base_model_path,
    "--proj_dir", "/kaggle/working/moe-out",
    "--train_data_file", "/kaggle/working/sft.h5",
    "--sft", "1",
    "--smoothing", "0.001",
    "--moe", "1",
    "--moe_experts", "4",
    "--moe_active", "2",
    "--layer_profile", "layerprofile/24_TEST_peft_moe.csv",
    "--vocab_size", "65536",
    "--ctx_len", "256",
    "--n_layer", "24",
    "--n_embd", "2048",
    "--dim_ffn", "8192",
    "--head_size_a", "64",
    "--my_testing", "x070",
    "--fla", "1",
    "--quant", "0",
    "--epoch_steps", "550",
    "--epoch_count", "1",
    "--epoch_begin", "0",
    "--epoch_save", "1",
    "--micro_bsz", "1",
    "--accumulate_grad_batches", "1",
    "--lr_init", "1e-4",
    "--lr_final", "1e-5",
    "--warmup_steps", "20",
    "--beta1", "0.9",
    "--beta2", "0.999",
    "--adam_eps", "1e-8",
    "--grad_cp", "1",
    "--strategy", "auto",
    "--devices", "1",
    "--precision", "bf16",
    "--accelerator", "gpu",
    "--random_seed", "1337"
]

print(f"Argv completo:\n{' '.join(argv)}")

log_path = "/kaggle/working/train_moe.log"
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

start_time = time.time()
train_rc = 0

with open(log_path, "w", encoding="utf-8") as log_file:
    proc = subprocess.Popen(
        argv,
        cwd="/kaggle/working/RWKV-LM-RLHF/main",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
        text=True,
        bufsize=1
    )
    for line in iter(proc.stdout.readline, ""):
        sys.stdout.write(line)
        sys.stdout.flush()
        log_file.write(line)
        log_file.flush()
    proc.stdout.close()
    try:
        train_rc = proc.wait(timeout=10800)
    except subprocess.TimeoutExpired:
        proc.kill()
        train_rc = -1
        msg = "\n[ERROR] Entrenamiento excedió timeout de 3 horas.\n"
        sys.stdout.write(msg)
        log_file.write(msg)

wall_s = time.time() - start_time
print(f"\nSubprocess terminado con rc={train_rc} en {wall_s:.2f}s")

print('[STEP] cell6 end', flush=True)


In [ ]:
print('[STEP] cell7 start', flush=True)
import os
import re
import json
import glob

print("=== Celda 5: Diagnóstico del Log ===")
log_path = "/kaggle/working/train_moe.log"
lines = []
if os.path.exists(log_path):
    with open(log_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

log_text = "".join(lines)

# Menciones MoE
n_menciones_moe = sum(1 for l in lines if any(w in l.lower() for w in ["moe", "expert", "router"]))

# Checkpoints
ckpt_files = sorted(glob.glob("/kaggle/working/moe-out/*.pth"))
ckpts = [{"path": p, "size_bytes": os.path.getsize(p)} for p in ckpt_files]

# Trainable params
trainable_params = None
m_tp = re.search(r"trainable params[:\s]+([0-9,]+)", log_text, re.IGNORECASE)
if not m_tp:
    m_tp = re.search(r"trainable parameters[:\s]+([0-9,]+)", log_text, re.IGNORECASE)
if m_tp:
    try:
        trainable_params = int(m_tp.group(1).replace(",", ""))
    except ValueError:
        pass

# Steps y Loss
step_nums = []
for m in re.finditer(r"(?:step|Epoch\s*\d+:[\s\d%|]*)\s*(\d+)\s*/\s*550", log_text, re.IGNORECASE):
    step_nums.append(int(m.group(1)))

steps_logueados = max(step_nums) if step_nums else 0

losses = []
n_loss_nan = 0
for l in lines:
    for m in re.finditer(r"loss[:=]\s*([0-9\.]+|nan)", l, re.IGNORECASE):
        val_str = m.group(1).lower()
        if val_str == "nan":
            n_loss_nan += 1
        else:
            try:
                losses.append(float(val_str))
            except ValueError:
                pass

if steps_logueados == 0 and losses:
    steps_logueados = len(losses)

loss_primero = losses[0] if losses else None
loss_ultimo = losses[-1] if losses else None
s_por_paso_real = round(wall_s / steps_logueados, 4) if (steps_logueados > 0 and "wall_s" in locals()) else None

ultimas_40 = [l.rstrip("\r\n") for l in lines[-40:]]

verdict = {
    "rc": train_rc if "train_rc" in locals() else None,
    "steps_logueados": steps_logueados,
    "n_loss_nan": n_loss_nan,
    "loss_primero": loss_primero,
    "loss_ultimo": loss_ultimo,
    "wall_s": round(wall_s, 2) if "wall_s" in locals() else None,
    "s_por_paso_real": s_por_paso_real,
    "n_menciones_moe": n_menciones_moe,
    "ckpts": ckpts,
    "trainable_params": trainable_params,
    "ultimas_40_lineas": ultimas_40
}

verdict_json_path = "/kaggle/working/moe_verdict.json"
with open(verdict_json_path, "w", encoding="utf-8") as f:
    json.dump(verdict, f, indent=2, ensure_ascii=False)

print(f"MOE_VERDICT: {json.dumps(verdict, indent=2, ensure_ascii=False)}")

print('[STEP] cell7 end', flush=True)


In [ ]:
print('[STEP] cell8 start', flush=True)
import os
import json
import math
import torch

print("=== Celda 6: Guarda de Sanidad ===")
verdict_path = "/kaggle/working/moe_verdict.json"
assert os.path.exists(verdict_path), f"No se encontró {verdict_path}"
with open(verdict_path, "r", encoding="utf-8") as f:
    v = json.load(f)

# (a) Pasos con loss finita al menos 10
finite_steps = v.get("steps_logueados", 0)
loss_first = v.get("loss_primero")
cond_a = (finite_steps >= 10) and (loss_first is not None and not math.isnan(loss_first))
print(f"[Sanidad A] Pasos con loss finita >= 10: {'PASS' if cond_a else 'FAIL'} (steps={finite_steps}, loss_primero={loss_first})")

# (b) n_loss_nan == 0
n_nan = v.get("n_loss_nan", 0)
cond_b = (n_nan == 0)
print(f"[Sanidad B] Sin losses NaN (n_loss_nan == 0): {'PASS' if cond_b else 'FAIL'} (n_loss_nan={n_nan})")

# (c) Existe al menos un .pth en moe-out
ckpts = v.get("ckpts", [])
cond_c = len(ckpts) > 0
print(f"[Sanidad C] Al menos un .pth en moe-out: {'PASS' if cond_c else 'FAIL'} (ckpts={len(ckpts)})")

# (d) Checkpoint contiene claves con expert/router
cond_d = False
d_detail = "sin checkpoints disponibles"
if ckpts:
    latest_ckpt_path = ckpts[-1]["path"]
    try:
        sd = torch.load(latest_ckpt_path, map_location="cpu")
        keys = list(sd.keys()) if isinstance(sd, dict) else []
        if "state_dict" in sd and isinstance(sd["state_dict"], dict):
            keys = list(sd["state_dict"].keys())
        expert_router_keys = [k for k in keys if "expert" in k.lower() or "router" in k.lower()]
        if expert_router_keys:
            cond_d = True
            d_detail = f"{len(expert_router_keys)} claves encontradas (ej. {expert_router_keys[:3]})"
        else:
            d_detail = f"0 claves expert/router entre {len(keys)} claves inspeccionadas"
    except Exception as e:
        d_detail = f"error torch.load en {latest_ckpt_path}: {e} (size={ckpts[-1].get('size_bytes')} bytes)"

print(f"[Sanidad D] Claves expert/router en checkpoint: {'PASS' if cond_d else 'FAIL'} ({d_detail})")

all_pass = cond_a and cond_b and cond_c and cond_d
print(f"\nVEREDICTO GLOBAL DE SANIDAD: {'PASS' if all_pass else 'FAIL'}")

print('[STEP] cell8 end', flush=True)
